# Libs 

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from numpy.typing import NDArray
from skimage import io
from tqdm.auto import tqdm

## A GPU está sendo utilizada? se sim, qual?

In [3]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 4060 Ti


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Paths, seeds e paletas

In [5]:
IMAGES_PATH = Path("./dados/10x/Images")
MASKS_PATH = Path("./dados/10x/Masks")
SEED_1 = 9
SEED_2 = 23
SEED_3 = 69
TRAIN_LIST = "./dados/train_files.txt"
TEST_LIST = "./dados/test_files.txt"
VALIDATION_LIST = "./dados/validation_files.txt"
PALETA = np.array([
    [0, 0, 0],        # 0 BKG fundo
    [248, 123, 168],  # 1 KER queratina
    [73, 0, 106],     # 2 EPI epiderme
    [236, 85, 157],   # 3 PAP derme papilar
    [181, 9, 130],    # 4 RET derme reticular
    [254, 246, 242],  # 5 HYP hipoderme
    [108, 0, 115],    # 6 GLD glândulas
    [216, 47, 148],   # 7 FOL folículos
    [145, 1, 122],    # 8 INF inflamação
    [127, 255, 255],  # 9 BCC
    [127, 255, 142],  # 10 SCC
    [255, 127, 127],  # 11 IEC
])

In [6]:
rng = np.random.default_rng(SEED_1)

## Utils

In [7]:
def read_list(path: str) -> list[str]:
    
    file_path = Path(path)
    
    file_list = file_path.read_text().splitlines()
    
    file_list_stripped = [row.strip() for row in file_list if row.strip() != ""]
    
    print(file_list_stripped[0:4])
    print(len(file_list_stripped))
    
    return file_list_stripped

In [8]:
def calculate_euclidean_distance(mask: NDArray[np.int32], paleta: NDArray) -> NDArray[np.float64]:
    
    calculated_distances = [((mask - color) ** 2).sum(axis= -1) for color in paleta]
    
    distances = np.stack(
        calculated_distances, 
        axis= -1,
    )
    
    sqrt_distances = np.sqrt(distances)
    
    return sqrt_distances
    

In [9]:
def masks_for_classes(mask_rgb: NDArray[np.uint8]) -> NDArray[np.uint8]:
    
    mask = mask_rgb.astype(np.int32)
    
    sqrt_distances = calculate_euclidean_distance(mask, PALETA)
    
    closest_classes = sqrt_distances.argmin(axis=-1).astype(np.uint8)

    return closest_classes

In [10]:
def load_images_list(images_names_list: list[str]) -> list[tuple[NDArray[np.uint8], NDArray[np.uint8]]]:
    
    images_list = []
    masks_list = []
    
    for name in tqdm(images_names_list, desc="Loading images"):
        
        image = io.imread(IMAGES_PATH/f"{name}.tif")
        images_list.append(image)
        
        mask = io.imread(MASKS_PATH/f"{name}.png")
        masks_list.append(masks_for_classes(mask))
        
    paired_lists = list(zip(images_list, masks_list))    
    
    return paired_lists

## Data

In [11]:
def random_cut(image, classes, cutsize=512):
    
    if image.shape[0] < cutsize or image.shape[1] < cutsize:
        
        missing_height = max(0, cutsize - image.shape[0])
        
        missing_width = max(0, cutsize - image.shape[1])
        
        image = np.pad(
            image,
            (
                (0, missing_height), 
                (0, missing_width),
                (0, 0)             
            ), 
            constant_values=255)
     
        
    if classes.shape[0] < cutsize or classes.shape[1] < cutsize:
        
        missing_height = max(0, cutsize - classes.shape[0])
        
        missing_width = max(0, cutsize - classes.shape[1])
        
        classes = np.pad(
            classes,
            (
                (0, missing_height), 
                (0, missing_width)          
            ), 
            constant_values=0)
        
    return image, classes
        
        

In [12]:
train_list = read_list(TRAIN_LIST)

['SCC_33', 'BCC_101', 'BCC_122', 'IEC_15']
231


In [13]:
test_list = read_list(TEST_LIST)

['BCC_10', 'BCC_116', 'SCC_4', 'SCC_42']
31


In [14]:
validation_list = read_list(VALIDATION_LIST)

['IEC_32', 'SCC_12', 'IEC_61', 'IEC_18']
28


In [15]:
train = load_images_list(train_list)
test = load_images_list(test_list)
validation = load_images_list(validation_list)


Loading images:   0%|          | 0/231 [00:00<?, ?it/s]

Loading images:   0%|          | 0/31 [00:00<?, ?it/s]

Loading images:   0%|          | 0/28 [00:00<?, ?it/s]

In [16]:
image, classes = random_cut(train[200][0], train[200][1])

In [17]:
image.shape, classes.shape

((512, 512, 3), (512, 512))

In [21]:
y = rng.integers(0, image.shape[0] - 512 + 1)
x = rng.integers(0, image.shape[1] - 512 + 1)

In [22]:
image_pad, classes_pad = random_cut(train[175][0], train[175][1])  # BCC_70
print("depois do preenchimento:", image_pad.shape, classes_pad.shape)

for i in range(5):
    y = rng.integers(0, image_pad.shape[0] - 512 + 1)
    x = rng.integers(0, image_pad.shape[1] - 512 + 1)

    image_crop = image_pad[y : y + 512, x : x + 512]
    classes_crop = classes_pad[y : y + 512, x : x + 512]

    print(f"{i}: y={y:3d}, x={x:3d} -> {image_crop.shape}, {classes_crop.shape}")

depois do preenchimento: (512, 630, 3) (512, 630)
0: y=  0, x= 50 -> (512, 512, 3), (512, 512)
1: y=  0, x=103 -> (512, 512, 3), (512, 512)
2: y=  0, x=114 -> (512, 512, 3), (512, 512)
3: y=  0, x= 34 -> (512, 512, 3), (512, 512)
4: y=  0, x= 13 -> (512, 512, 3), (512, 512)
